# Start Trek — DQN

Agent DQN entraîné à poser le module de `LunarLander-v3`.

Objectif : score moyen ≥ 200 sur 100 épisodes consécutifs, sans exploration.

## Sommaire

1. Imports et configuration
2. Environnement
3. Raison de fin d'épisode
4. Boucle d'épisode et logs
5. Baseline : agent random
6. Hyperparamètres
7. Réseau Q et loss
8. Replay buffer
9. Epsilon-greedy
10. Target network et update

## 1. Imports et configuration

Toutes les seeds sont regroupées ici pour que les résultats soient reproductibles.

In [ ]:
import json
import time
from collections import Counter
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

In [ ]:
ENV_ID = "LunarLander-v3"
SEED = 0
SEEDS = [0, 1, 2]

RESULTS_DIR = Path("results/dqn")
MODELS_DIR = Path("models")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)

## 2. Environnement

Deux façons de créer l'environnement :

- **entraînement / évaluation** : `render_mode=None`, aucun rendu, c'est beaucoup plus rapide ;
- **visualisation** : `render_mode="human"`, uniquement pour regarder l'agent jouer.

La taille de l'état et le nombre d'actions sont lus depuis l'environnement, pas écrits en dur.

In [ ]:
def make_env(render_mode=None):
    return gym.make(ENV_ID, render_mode=render_mode)


env = make_env()
OBS_DIM = env.observation_space.shape[0]
N_ACTIONS = env.action_space.n
MAX_STEPS = env.spec.max_episode_steps

print(f"Taille de l'état : {OBS_DIM}")
print(f"Nombre d'actions : {N_ACTIONS}")
print(f"Steps max par épisode : {MAX_STEPS}")

## 3. Raison de fin d'épisode

Gymnasium renvoie seulement `terminated` et `truncated`, pas la raison exacte. On la retrouve à partir de l'état interne de l'environnement (`env.unwrapped`), avec les mêmes conditions que le code source de LunarLander :

| Label | Type | Condition |
| --- | --- | --- |
| `crash` | terminated | le corps du module a touché le sol (`game_over`) |
| `out_of_viewport` | terminated | `abs(x) >= 1` |
| `at_rest` | terminated | le module ne bouge plus (`lander.awake == False`) |
| `truncated` | truncated | limite de steps atteinte |

`crash` est testé en premier : si le module touche le sol et sort de l'écran au même step, c'est le contact qui a déclenché la fin.

In [ ]:
END_REASONS = ["crash", "out_of_viewport", "at_rest", "truncated"]


def get_end_reason(env, state, terminated, truncated):
    if truncated and not terminated:
        return "truncated"
    lander_env = env.unwrapped
    if lander_env.game_over:
        return "crash"
    if abs(state[0]) >= 1.0:
        return "out_of_viewport"
    if not lander_env.lander.awake:
        return "at_rest"
    return "unknown"

## 4. Boucle d'épisode et logs

`run_episode` joue un épisode complet avec une politique donnée (une fonction `state -> action`) et renvoie un log :

| Champ | Contenu |
| --- | --- |
| `episode` | numéro de l'épisode |
| `reward` | reward totale |
| `length` | nombre de steps |
| `end_reason` | raison de fin (section 3) |
| `global_step` | nombre total de steps joués depuis le début |
| `wall_time` | temps écoulé depuis le début (secondes) |

Ce format est le même dans `ppo.ipynb`, pour pouvoir comparer les deux agents.

In [ ]:
def run_episode(env, policy, seed=None, episode=0, global_step=0, start_time=None):
    if start_time is None:
        start_time = time.time()
    state, _ = env.reset(seed=seed)
    total_reward = 0.0
    length = 0
    done = False

    while not done:
        action = policy(state)
        state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        length += 1
        done = terminated or truncated

    return {
        "episode": episode,
        "reward": float(total_reward),
        "length": length,
        "end_reason": get_end_reason(env, state, terminated, truncated),
        "global_step": global_step + length,
        "wall_time": time.time() - start_time,
    }


def summarize(logs):
    rewards = np.array([log["reward"] for log in logs])
    reasons = Counter(log["end_reason"] for log in logs)
    return {
        "n_episodes": len(logs),
        "mean": float(rewards.mean()),
        "std": float(rewards.std()),
        "min": float(rewards.min()),
        "max": float(rewards.max()),
        "end_reasons": {reason: reasons.get(reason, 0) for reason in END_REASONS},
    }

## 5. Baseline : agent random

Avant d'entraîner quoi que ce soit, on joue 100 épisodes en choisissant les actions au hasard. Ça donne un score de référence et ça vérifie que les logs marchent.

Chaque épisode `i` utilise la seed `SEED + i` : la baseline est identique à chaque exécution.

In [ ]:
N_BASELINE_EPISODES = 100

env.action_space.seed(SEED)
random_policy = lambda state: env.action_space.sample()

baseline_logs = []
global_step = 0
start_time = time.time()
for i in range(N_BASELINE_EPISODES):
    log = run_episode(env, random_policy, seed=SEED + i, episode=i,
                      global_step=global_step, start_time=start_time)
    global_step = log["global_step"]
    baseline_logs.append(log)

baseline = summarize(baseline_logs)
print(f"Score moyen : {baseline['mean']:.1f} ± {baseline['std']:.1f}")
print(f"Min / max   : {baseline['min']:.1f} / {baseline['max']:.1f}")
print(f"Raisons de fin : {baseline['end_reasons']}")

In [ ]:
rewards = [log["reward"] for log in baseline_logs]
reasons = baseline["end_reasons"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(rewards, bins=20, color="tab:gray", edgecolor="black")
axes[0].axvline(baseline["mean"], color="tab:red", linestyle="--", label=f"moyenne {baseline['mean']:.1f}")
axes[0].axvline(200, color="tab:green", linestyle="--", label="objectif 200")
axes[0].set_title("Agent random : distribution des scores")
axes[0].set_xlabel("Reward totale")
axes[0].set_ylabel("Nombre d'épisodes")
axes[0].legend()

axes[1].bar(reasons.keys(), reasons.values(), color="tab:gray", edgecolor="black")
axes[1].set_title("Agent random : raisons de fin")
axes[1].set_ylabel("Nombre d'épisodes")

plt.tight_layout()
plt.show()

In [ ]:
with open(RESULTS_DIR / "baseline_random.json", "w") as f:
    json.dump({"summary": baseline, "episodes": baseline_logs}, f, indent=2)

**Analyse :** _à compléter après exécution (score moyen, raison de fin dominante, ce que ça dit du problème)._

## 6. Hyperparamètres

Tous les hyperparamètres du DQN sont regroupés ici. Ce sont des valeurs de départ, réglées ensuite pendant le tuning.

| Paramètre | Valeur | Raison |
| --- | --- | --- |
| `gamma` | 0.99 | un atterrissage prend plusieurs centaines de steps, le +100 final doit remonter jusqu'aux premières actions |
| `lr` | 5e-4 | valeur classique avec Adam sur un petit MLP |
| `batch_size` | 64 | assez gros pour lisser le gradient, assez petit pour rester rapide |
| `buffer_capacity` | 100 000 | une centaine d'épisodes : assez de diversité sans garder des expériences trop vieilles |
| `learning_starts` | 1 000 | on ne s'entraîne pas sur un buffer presque vide |
| `hidden_sizes` | (128, 128) | état de taille 8, 4 actions : un petit réseau suffit |
| `eps_start` / `eps_end` | 1.0 / 0.01 | 100 % d'exploration au début, presque plus à la fin |
| `eps_decay_steps` | 100 000 | voir section 9 |
| `tau` | 0.005 | voir section 10 |

In [ ]:
CONFIG = {
    "gamma": 0.99,
    "lr": 5e-4,
    "batch_size": 64,
    "buffer_capacity": 100_000,
    "learning_starts": 1_000,
    "hidden_sizes": (128, 128),
    "eps_start": 1.0,
    "eps_end": 0.01,
    "eps_decay_steps": 100_000,
    "tau": 0.005,
}

## 7. Réseau Q et loss

L'état est continu : impossible de stocker une Q-table. On utilise un **MLP** qui prend l'état en entrée et sort **une Q-value par action** :

`état (8) → Linear → ReLU → Linear → ReLU → Linear → Q-values (4)`

Avec une sortie par action, un seul passage dans le réseau donne les 4 Q-values d'un état. Choisir la meilleure action, c'est juste un `argmax`.

**Loss : MSE.** Pour une expérience `(s, a, r, s', done)` on construit une cible avec l'équation de Bellman :

- `target = r` si l'épisode est vraiment fini (`done`) ;
- `target = r + γ · max_a' Q_target(s', a')` sinon.

La loss est l'erreur quadratique moyenne entre la Q-value prédite pour l'action jouée, `Q(s, a)`, et cette cible. Seule la Q-value de l'action jouée est corrigée : on ne sait rien des 3 autres.

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden_sizes=(128, 128)):
        super().__init__()
        layers = []
        in_size = obs_dim
        for size in hidden_sizes:
            layers += [nn.Linear(in_size, size), nn.ReLU()]
            in_size = size
        layers.append(nn.Linear(in_size, n_actions))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


q_net = QNetwork(OBS_DIM, N_ACTIONS, CONFIG["hidden_sizes"]).to(DEVICE)
print(q_net)
print(f"Paramètres : {sum(p.numel() for p in q_net.parameters())}")

## 8. Replay buffer

Deux steps qui se suivent sont presque identiques. Apprendre dessus dans l'ordre rend l'entraînement instable. Le replay buffer stocke les expériences et on apprend sur des **batches tirés au hasard**.

- **capacité fixe** : tableaux numpy pré-alloués, utilisés en cercle. Une fois plein, la nouvelle expérience écrase la plus vieille.
- **`sample`** renvoie des tenseurs groupés par élément (`states`, `actions`, ...), prêts pour le réseau.
- **`len`** donne le nombre d'expériences stockées, pour savoir quand commencer à apprendre.

**Ce qu'on stocke dans `done` : `terminated` uniquement.** Quand l'épisode est `truncated`, le module aurait pu continuer : la cible doit garder `γ · max Q(s')`. Si on mettait `done = True`, on apprendrait à l'agent qu'il n'y a plus rien à gagner après la limite de temps, ce qui est faux.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity, obs_dim):
        self.capacity = capacity
        self.states = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.next_states = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.dones = np.zeros(capacity, dtype=np.float32)
        self.pos = 0
        self.size = 0

    def push(self, state, action, reward, next_state, done):
        self.states[self.pos] = state
        self.actions[self.pos] = action
        self.rewards[self.pos] = reward
        self.next_states[self.pos] = next_state
        self.dones[self.pos] = done
        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        idx = np.random.randint(0, self.size, size=batch_size)
        return (
            torch.as_tensor(self.states[idx], device=DEVICE),
            torch.as_tensor(self.actions[idx], device=DEVICE),
            torch.as_tensor(self.rewards[idx], device=DEVICE),
            torch.as_tensor(self.next_states[idx], device=DEVICE),
            torch.as_tensor(self.dones[idx], device=DEVICE),
        )

    def __len__(self):
        return self.size

Vérification : un buffer de capacité 3 qui reçoit 5 expériences ne doit garder que les 3 dernières.

In [ ]:
test_buffer = ReplayBuffer(capacity=3, obs_dim=OBS_DIM)
for i in range(5):
    test_buffer.push(np.full(OBS_DIM, i), i % N_ACTIONS, float(i), np.full(OBS_DIM, i + 1), False)

print(f"Taille : {len(test_buffer)}")
print(f"Rewards stockées : {sorted(test_buffer.rewards.tolist())}")

states, actions, rewards, next_states, dones = test_buffer.sample(4)
print(f"Shapes : {tuple(states.shape)} {tuple(actions.shape)} {tuple(rewards.shape)} {tuple(next_states.shape)} {tuple(dones.shape)}")

## 9. Epsilon-greedy

- avec une probabilité `ε` : action au hasard (exploration) ;
- sinon : action avec la plus grande Q-value (exploitation).

**Décroissance choisie : linéaire, par step**, de `eps_start` à `eps_end` sur `eps_decay_steps` steps, puis constante.

- **par step plutôt que par épisode** : la longueur des épisodes change énormément (un crash rapide fait ~70 steps, un atterrissage plusieurs centaines). Compter en steps relie l'exploration à la quantité d'expérience réellement vécue.
- **linéaire plutôt qu'exponentielle** : le temps passé à explorer se lit directement dans un seul paramètre. Une exponentielle descend très vite au début, au moment où l'agent ne sait encore rien.

En évaluation, `ε = 0` : l'agent prend toujours la meilleure action.

In [ ]:
def get_epsilon(step, eps_start=CONFIG["eps_start"], eps_end=CONFIG["eps_end"],
                decay_steps=CONFIG["eps_decay_steps"]):
    progress = min(step / decay_steps, 1.0)
    return eps_start + progress * (eps_end - eps_start)


def select_action(q_net, state, epsilon):
    if np.random.random() < epsilon:
        return np.random.randint(N_ACTIONS)
    with torch.no_grad():
        state_t = torch.as_tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        return int(q_net(state_t).argmax(dim=1).item())


steps = np.arange(0, 150_000, 1_000)
plt.figure(figsize=(6, 3))
plt.plot(steps, [get_epsilon(s) for s in steps])
plt.title("Décroissance de ε")
plt.xlabel("Step")
plt.ylabel("ε")
plt.grid(alpha=0.3)
plt.show()

## 10. Target network et update

Si la cible est calculée avec le réseau qu'on entraîne, elle bouge à chaque update : on vise une cible qui se déplace et l'entraînement devient instable. Le **target network** est une copie du réseau principal, utilisée seulement pour calculer les cibles, et mise à jour plus lentement.

Deux stratégies possibles :

- **hard update** : on recopie tous les poids tous les N steps. La cible reste fixe pendant N steps, puis change d'un coup.
- **soft update** : à chaque step, `θ_target ← τ · θ + (1 − τ) · θ_target`. La cible suit le réseau principal en douceur.

**Choix : soft update avec `τ = 0.005`.** La cible ne fait jamais de saut brusque, ce qui évite les pics de loss qu'on a juste après une copie en hard update. Avec `τ = 0.005`, le target network « oublie » ses anciens poids sur environ `1/τ = 200` steps, l'équivalent d'un hard update toutes les quelques centaines de steps. La comparaison hard vs soft sera faite dans l'expérience comparative.

In [ ]:
def soft_update(target_net, q_net, tau):
    with torch.no_grad():
        for target_param, param in zip(target_net.parameters(), q_net.parameters()):
            target_param.mul_(1 - tau).add_(tau * param)


def hard_update(target_net, q_net):
    target_net.load_state_dict(q_net.state_dict())


def compute_targets(target_net, rewards, next_states, dones, gamma):
    with torch.no_grad():
        next_q = target_net(next_states).max(dim=1).values
    return rewards + gamma * next_q * (1 - dones)


def dqn_update(q_net, target_net, optimizer, buffer, batch_size, gamma):
    states, actions, rewards, next_states, dones = buffer.sample(batch_size)
    q_values = q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
    targets = compute_targets(target_net, rewards, next_states, dones, gamma)
    loss = nn.functional.mse_loss(q_values, targets)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

Vérifications :

1. une expérience terminée (`done = 1`) doit avoir une cible égale à `r` ;
2. un update complet sur un buffer rempli d'expériences random doit renvoyer une loss finie.

In [ ]:
target_net = QNetwork(OBS_DIM, N_ACTIONS, CONFIG["hidden_sizes"]).to(DEVICE)
hard_update(target_net, q_net)

rewards_t = torch.tensor([100.0, 1.0])
next_states_t = torch.zeros((2, OBS_DIM))
dones_t = torch.tensor([1.0, 0.0])
print(f"Cibles : {compute_targets(target_net, rewards_t, next_states_t, dones_t, CONFIG['gamma']).tolist()}")

buffer = ReplayBuffer(CONFIG["buffer_capacity"], OBS_DIM)
state, _ = env.reset(seed=SEED)
for _ in range(CONFIG["learning_starts"]):
    action = env.action_space.sample()
    next_state, reward, terminated, truncated, _ = env.step(action)
    buffer.push(state, action, reward, next_state, terminated)
    state = next_state if not (terminated or truncated) else env.reset()[0]

optimizer = torch.optim.Adam(q_net.parameters(), lr=CONFIG["lr"])
loss = dqn_update(q_net, target_net, optimizer, buffer, CONFIG["batch_size"], CONFIG["gamma"])
soft_update(target_net, q_net, CONFIG["tau"])
print(f"Buffer : {len(buffer)} expériences, loss du premier update : {loss:.3f}")